In [15]:

import os
import re
import pandas as pd
import platform
from datetime import datetime, timedelta
from scipy.io import wavfile
import numpy as np
from tqdm import tqdm
import sys
print(sys.executable)
import ast
import platform
from pathlib import Path

if platform.system() == "Windows":
    base_raw = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data")
    base_processed = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio")
    base_das = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox")
else:
    base_raw = Path("/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/")
    base_processed = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data")
    base_das = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Pre_processing/das/models/all_vox")

print("I'm at: ",base_raw)
print(base_das)

c:\Users\gg3065\code_projects\gerbil_vocalization_analysis\.venv\Scripts\python.exe
I'm at:  \\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data
\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox


In [14]:
import pandas as pd
from pathlib import Path

# === INPUT / OUTPUT FOLDERS ===
input_folder = Path(fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10")
output_folder = Path(fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_unified")

output_folder.mkdir(parents=True, exist_ok=True)

# === DEFINE MAPPING ===
mapping = {
    "dense-stack": "stack",
    "dfm-stack": "stack",

}

# === PROCESS ALL CSV FILES ===
for csv_file in input_folder.glob("*.csv"):
    print(f"Processing: {csv_file.name}")

    df = pd.read_csv(csv_file)

    # --- Apply mapping ---
    df["name"] = df["name"].apply(lambda x: mapping.get(x, x))

    # --- Save to new folder ---
    out_path = output_folder / csv_file.name
    df.to_csv(out_path, index=False)

print("Done!")

Processing: channel_0_61_alarm_annotations.csv
Processing: channel_3_67_alarm_annotations.csv
Processing: channel_4_12__alarm_annotations.csv
Processing: channel_4_138_annotations.csv
Processing: channel_4_156_complex_annotations.csv
Processing: channel_6_61_complex_annotations.csv
Processing: exp_235_channel_30_file_024__half-mnt_annotations.csv
Processing: exp_235_channel_30_file_024__mnt_annotations.csv
Processing: exp_235_channel_30_file_040__B_dense-stack_annotations.csv
Processing: exp_235_channel_30_file_040__B_half-mnt_annotations.csv
Processing: exp_235_channel_30_file_040__C_half-mnt_dense-stack_annotations.csv
Processing: exp_235_channel_30_file_040__C_mnt_annotations.csv
Processing: exp_235_channel_30_file_040__D_half-mnt_dense-stack_annotations.csv
Processing: exp_235_channel_30_file_040__half-mnt_annotations.csv
Processing: exp_235_channel_30_file_040__I_half-mnt_annotations.csv
Processing: exp_235_channel_30_file_040__J_half-mnt_annotations.csv
Processing: exp_235_channe

In [22]:
exp = 518

def get_experiment_month(exp):
    if 97 <= exp <= 116:
        return "2024_12"
    elif 235 <= exp <= 237:
        return "2025_03"
    elif 272 <= exp < 278:
        return "2025_07"
    elif 332 <= exp < 344:
        return "2025_10"
    elif exp >= 491:
        return "2026_02"

    else:
        raise ValueError(f"Unknown experiment range for {exp}")
month_folder = get_experiment_month(exp)


# Build paths
folder_path_raw_wavs = base_raw / f"experiment_{exp}" / "concatenated_data_cam_mic_sync" #/ "wavs"
folder_path_sync = base_raw / f"experiment_{exp}" / "concatenated_data_cam_mic_sync"
folder_path_averaged_wavs = base_processed / "Audio" / month_folder / str(exp) / "Averaged_wavs_w_annotations"
folder_path_Audio = base_processed / "Audio" / month_folder / str(exp)

# Optional: print to verify
print(folder_path_raw_wavs)
print(folder_path_averaged_wavs)
print(folder_path_Audio)


# --- Parameters ----------------------------------------------------
#arena_wav_folder = folder_path_averaged_wavs  
sample_rate_expected = 125000
#output_folder = folder_path_Audio  #  output path for CSVs

time_window_overlapping_calls_sec = 0.015
inter_call_interval_within_bout_threshold = 10.0  # seconds
min_calls_per_bout = 5 


# ##############################################################################
# load sync file
# ##############################################################################

def load_sync_file(exp, folder_path_sync):
    sync_path = Path(folder_path_sync) / "sync.csv"
    
    if not sync_path.exists():
        raise FileNotFoundError(f"Sync file not found: {sync_path}")

    sync_df = pd.read_csv(sync_path)
    print(list(sync_df.columns))
    
    # parse the time lists
    sync_df['timestamp'] = sync_df['timestamp'].apply(ast.literal_eval)
   
    # chaneg to date object
    sync_df['start_time'] = pd.to_datetime(sync_df['timestamp'].str[0])
    start_time_experiment = sync_df.iloc[0]['start_time']
    print(f"Experiment {exp} started at: {start_time_experiment}")

    return start_time_experiment, sync_df


# Load sync file 
start_time_experiment,sync_df = load_sync_file(exp,folder_path_sync)
#sync_df.head()


/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_518/concatenated_data_cam_mic_sync
/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/518/Averaged_wavs_w_annotations
/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/518
['video', 'audio', 'timestamp']
Experiment 518 started at: 2026-02-27 14:47:13


In [ ]:
# 0)------ Avergae audio data ----- > average two microphones for each arena
# ##############################################################################

def get_channel_mapping(exp):

    if exp < 272:
        return {
            "10": [0, 1],
            "20": [2, 3],
            "30": [4, 5],
        }
    else: # => 272
        return {
            "10": [2, 3],
            "20": [4, 5],
            "30": [0, 1],
        }


def average_microphone_pairs(exp, input_folder, output_folder):

    input_folder = Path(input_folder)
    output_folder = Path(output_folder)

    # Create the output folder if it does not already exist
    output_folder.mkdir(parents=True, exist_ok=True)

    # get the correct channel mapping for this experiment
    channel_mapping = get_channel_mapping(exp)

    print(f"\n--- Processing experiment {exp} ---")
    print(f"Reading raw WAV files from: {input_folder}")
    print(f"Saving averaged WAV files to: {output_folder}")
    print("Using channel mapping:")

    for virtual_ch, real_pair in channel_mapping.items():
        print(f"  virtual channel {virtual_ch} <- real channels {real_pair}")

    # --------------------------------------------------
    # STEP 1: Find all file numbers
    #
    # We use channel_00 files as a reference to find which
    # file chunks exist: 001, 002, 003, ...
    # --------------------------------------------------

    file_nums = []

    # Compile the regex once
    pattern = re.compile(r"channel_00_file_(\d+)\.wav")

    for file_path in input_folder.iterdir():
        match = pattern.match(file_path.name)

        if match:
            # Extract file number, e.g. "001" -> 1
            file_num = int(match.group(1))
            file_nums.append(file_num)

    # Sort file numbers so they are processed in order
    file_nums = sorted(file_nums)

    print(f"Found {len(file_nums)} file chunks")

    # Counters for summary at the end
    n_saved = 0
    n_missing = 0
    n_shape_mismatch = 0
    n_rate_mismatch = 0

    # --------------------------------------------------
    # STEP 2: Loop over each file number
    # --------------------------------------------------

    for file_num in file_nums:

        # Format with leading zeros: 1 -> "001"
        file_num_str = f"{file_num:03d}"

        # --------------------------------------------------
        # STEP 3: For this file number, create each virtual channel
        # --------------------------------------------------

        for virtual_ch, real_pair in channel_mapping.items():

            # real_pair is something like [2, 3]
            ch1, ch2 = real_pair
            #print(f"\nFile {file_num_str}: averaging channels {ch1} and {ch2} -> virtual channel {virtual_ch}")

            # Build the full paths to the two real microphone files
            path1 = folder_path_raw_wavs / f"channel_{ch1:02d}_file_{file_num_str}.wav"
            path2 = folder_path_raw_wavs / f"channel_{ch2:02d}_file_{file_num_str}.wav"

            # --------------------------------------------------
            # STEP 4: Check that both files exist
            # --------------------------------------------------

            if not path1.exists() or not path2.exists():
                print("  Skipping because one or both files are missing")
                print(f"  Expected file: {path1.name}")
                print(f"  Expected file: {path2.name}")
                n_missing += 1
                continue

            # --------------------------------------------------
            # STEP 5: Read both WAV files
            # --------------------------------------------------

            rate1, data1 = wavfile.read(path1) #data = waveform samples as a numpy array
            rate2, data2 = wavfile.read(path2)

            # --------------------------------------------------
            # STEP 6: Check that the two files are compatible
            # --------------------------------------------------

            if rate1 != rate2:
                print(f"  Skipping because sample rates differ: {rate1} vs {rate2}")
                n_rate_mismatch += 1
                continue

            if data1.shape != data2.shape:
                print(f"  Skipping because shapes differ: {data1.shape} vs {data2.shape}")
                n_shape_mismatch += 1
                continue

            # --------------------------------------------------
            # STEP 7: Average the two signals
            # We convert to float32
            # --------------------------------------------------

            data1 = data1.astype(np.float32, copy=False)
            data2 = data2.astype(np.float32, copy=False)

            avg_data = (data1 + data2) / 2.0

            # --------------------------------------------------
            # STEP 8: Save the averaged output
            # --------------------------------------------------

            out_path = output_folder / f"channel_{virtual_ch}_file_{file_num_str}.wav"

            wavfile.write(out_path, rate1, avg_data)

            #print(f"  Saved: {out_path.name}")
            n_saved += 1

    # --------------------------------------------------
    # STEP 9: Print summary
    # --------------------------------------------------

    print("\n--- Done ---")
    print(f"Saved files: {n_saved}")
    print(f"Skipped because of missing files: {n_missing}")
    print(f"Skipped because of shape mismatch: {n_shape_mismatch}")
    print(f"Skipped because of sample rate mismatch: {n_rate_mismatch}")

average_microphone_pairs(exp, folder_path_raw_wavs, folder_path_averaged_wavs)

In [ ]:
# plot DAS vox
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import spectrogram

# ============================================================
# SETTINGS
# ============================================================

csv_folder = base_das / "TRAINING_data_11"
audio_folder = base_das / "TRAINING_data_11"   # can contain .wav and/or .npz
output_folder = base_das / "TRAINING_data_11_viz"

# csv_folder = base_processed / "2026_02" / "547" / "Averaged_wavs_w_annotations_unified"
# audio_folder = csv_folder   # can contain .wav and/or .npz
# output_folder = base_processed / "2026_02" / "547" / "das_viz_unified"

output_folder.mkdir(parents=True, exist_ok=True)
print(audio_folder)

# CSV column names
label_col = "name"
start_col = "start_seconds"
stop_col = "stop_seconds"

# Spectrogram settings
buffer_sec = 0
nperseg = 512
noverlap = 256
spec_min_val = -8.0
spec_max_val = -5.0

# Plot settings
max_examples_per_label = None   # e.g. 200, or None for all
examples_per_page = 100
cols = 10
figsize_per_panel = 1.5


# ============================================================
# HELPERS
# ============================================================

def is_real_event_row(df: pd.DataFrame) -> pd.Series:
    """
    Keep only rows that contain a real event.
    Removes DAS template/empty rows.
    """
    start = pd.to_numeric(df[start_col], errors="coerce")
    stop = pd.to_numeric(df[stop_col], errors="coerce")

    return (
        start.notna()
        & stop.notna()
        & (stop > start)
        & (start > 0)
    )


def clean_label(label: str) -> str:
    return str(label).strip()


def find_matching_audio(csv_path: Path) -> Path | None:
    """
    Match:
      exp_272_channel_30_file_063_annotations.csv
    to either:
      exp_272_channel_30_file_063.wav
      exp_272_channel_30_file_063.npz
    """
    stem = csv_path.stem

    if stem.endswith("_annotations"):
        stem = stem[:-12]   # remove "_annotations"

    wav_path = audio_folder / f"{stem}.wav"
    if wav_path.exists():
        return wav_path

    npz_path = audio_folder / f"{stem}.npz"
    if npz_path.exists():
        return npz_path

    return None


def load_audio_file(audio_path: Path):
    """
    Load either .wav or .npz audio.

    Returns
    -------
    fs : int or float
    audio : np.ndarray
    """
    suffix = audio_path.suffix.lower()

    if suffix == ".wav":
        fs, audio = wavfile.read(audio_path)
        return fs, audio

    elif suffix == ".npz":
        data = np.load(audio_path, allow_pickle=True)

        print(f"[info] npz keys in {audio_path.name}: {data.files}")

        possible_audio_keys = ["audio", "wav", "waveform", "x", "samples"]
        possible_fs_keys = ["fs", "sr", "samplerate", "sample_rate"]

        audio = None
        fs = None

        for k in possible_audio_keys:
            if k in data.files:
                audio = data[k]
                break

        for k in possible_fs_keys:
            if k in data.files:
                fs = data[k]
                break

        if audio is None:
            raise ValueError(
                f"Could not find audio array in {audio_path.name}. "
                f"Available keys: {data.files}"
            )

        if fs is None:
            raise ValueError(
                f"Could not find sample rate in {audio_path.name}. "
                f"Available keys: {data.files}"
            )

        if isinstance(fs, np.ndarray):
            fs = fs.item()

        return fs, audio

    else:
        raise ValueError(f"Unsupported audio type: {audio_path}")


def make_spec(signal: np.ndarray, fs: int):
    """
    Return an AVA-style normalized log spectrogram for plotting.
    """
    if signal.ndim > 1:
        signal = signal[:, 0]

    if len(signal) < nperseg:
        signal = np.pad(signal, (0, nperseg - len(signal)))

    f, t, Sxx = spectrogram(
        signal,
        fs=fs,
        nperseg=nperseg,
        noverlap=noverlap,
        scaling="spectrum",
        mode="magnitude",
    )

    Sxx = np.log(Sxx + 1e-12)
    Sxx = (Sxx - spec_min_val) / (spec_max_val - spec_min_val)
    Sxx = np.clip(Sxx, 0.0, 1.0)
    return f, t, Sxx


# ============================================================
# STEP 1: COLLECT ALL CALLS BY LABEL
# ============================================================

calls_by_label = {}

csv_files = sorted(csv_folder.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files")

for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
    except Exception as e:
        print(f"[skip] could not read {csv_file.name}: {e}")
        continue

    needed = {label_col, start_col, stop_col}
    if not needed.issubset(df.columns):
        print(f"[skip] missing columns in {csv_file.name}")
        continue

    df = df[is_real_event_row(df)].copy()
    if len(df) == 0:
        continue

    audio_path = find_matching_audio(csv_file)
    if audio_path is None:
        print(f"[skip] no matching audio for {csv_file.name}")
        continue

    df[start_col] = pd.to_numeric(df[start_col], errors="coerce")
    df[stop_col] = pd.to_numeric(df[stop_col], errors="coerce")

    for _, row in df.iterrows():
        label = clean_label(row[label_col])

        event = {
            "csv_file": csv_file,
            "audio_file": audio_path,
            "label": label,
            "start": float(row[start_col]),
            "stop": float(row[stop_col]),
            "duration": float(row[stop_col] - row[start_col]),
        }

        calls_by_label.setdefault(label, []).append(event)

print("\nCollected calls by label:")
for label, events in sorted(calls_by_label.items()):
    print(f"  {label}: {len(events)} calls")


# ============================================================
# STEP 2: PLOT EACH LABEL
# ============================================================

for label, events in sorted(calls_by_label.items()):
    if max_examples_per_label is not None:
        events = events[:max_examples_per_label]

    if len(events) == 0:
        continue

    print(f"\nPlotting label '{label}' with {len(events)} calls")

    label_out = output_folder / label
    label_out.mkdir(parents=True, exist_ok=True)

    # Group by audio file so we don't reread the same file too many times
    events_by_audio = {}
    for ev in events:
        events_by_audio.setdefault(ev["audio_file"], []).append(ev)

    all_specs = []

    for audio_path, audio_events in events_by_audio.items():
        try:
            fs, audio = load_audio_file(audio_path)
        except Exception as e:
            print(f"[skip audio] {audio_path.name}: {e}")
            continue

        if audio.ndim > 1:
            audio = audio[:, 0]

        n_samples = len(audio)

        for ev in audio_events:
            start = max(0, ev["start"] - buffer_sec)
            stop = ev["stop"] + buffer_sec

            s0 = int(round(start * fs))
            s1 = int(round(stop * fs))
            s1 = min(s1, n_samples)

            if s1 <= s0:
                continue

            segment = audio[s0:s1]

            try:
                f, t, Sxx = make_spec(segment, fs)
            except Exception as e:
                print(f"[skip spec] {audio_path.name}: {e}")
                continue

            all_specs.append({
                "spec": Sxx,
                "event": ev,
            })

    if len(all_specs) == 0:
        print(f"[skip label] no spectrograms made for {label}")
        continue

    num_pages = math.ceil(len(all_specs) / examples_per_page)

    for page_idx in range(num_pages):
        start_i = page_idx * examples_per_page
        end_i = min((page_idx + 1) * examples_per_page, len(all_specs))
        page_specs = all_specs[start_i:end_i]

        n = len(page_specs)
        ncols = cols
        nrows = math.ceil(n / ncols)

        fig, axes = plt.subplots(
            nrows, ncols,
            figsize=(ncols * figsize_per_panel, nrows * figsize_per_panel)
        )

        if nrows == 1 and ncols == 1:
            axes = np.array([[axes]])
        elif nrows == 1:
            axes = np.array([axes])
        elif ncols == 1:
            axes = axes[:, np.newaxis]

        axes = axes.flatten()

        for ax, item in zip(axes, page_specs):
            Sxx = item["spec"]
            ev = item["event"]

            ax.imshow(
                Sxx,
                aspect="auto",
                origin="lower",
                interpolation="nearest",
                vmin=0.0,
                vmax=1.0,
            )
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(ev["audio_file"].name, fontsize=4)

        for ax in axes[len(page_specs):]:
            ax.axis("off")

        fig.suptitle(
            f"{label} — page {page_idx + 1}/{num_pages} — n={len(page_specs)}",
            fontsize=14
        )
        fig.tight_layout()

        out_path = label_out / f"{label}_page_{page_idx + 1:03d}.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved {num_pages} page(s) for label '{label}' to {label_out}")

print("\nDone.")

\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\2026_02\547\Averaged_wavs_w_annotations_unified
Found 60 CSV files

Collected calls by label:
  alarm: 137 calls
  high-freq: 1557 calls
  mnt: 215 calls
  newborn: 4760 calls
  noise: 4747 calls
  stack: 2337 calls
  warble: 877 calls

Plotting label 'alarm' with 137 calls
Saved 2 page(s) for label 'alarm' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\2026_02\547\das_viz_unified\alarm

Plotting label 'high-freq' with 1557 calls
Saved 16 page(s) for label 'high-freq' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\2026_02\547\das_viz_unified\high-freq

Plotting label 'mnt' with 215 calls
Saved 3 page(s) for label 'mnt' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\2026_02\547\das_viz_unified\mnt

Plotting label 'newborn' with 4760 calls
Saved 48 page(s) for label 'newborn' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data\Audio\2026_02\547\das_viz_unif

In [ ]:
# ============================================================
# stacks with duration
# ============================================================

import math
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import spectrogram
from scipy.ndimage import binary_dilation, label as nd_label, median_filter
from skimage import feature, filters, morphology, restoration

if platform.system() == "Windows":
    base_das_local = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox")
else:
    base_das_local = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Pre_processing/das/models/all_vox")

csv_folder = base_das_local / "TRAINING_data_10"
audio_folder = csv_folder
output_folder = base_das_local / "TRAINING_data_10_viz_harmon_fullband_cleanaudio"
output_folder.mkdir(parents=True, exist_ok=True)
preview_folder = output_folder / "_point_previews"
preview_folder.mkdir(parents=True, exist_ok=True)
print(audio_folder)

target_labels = {"mnt", "dense-stack", "dfm-stack"}
label_col = "name"
start_col = "start_seconds"
stop_col = "stop_seconds"

buffer_sec = 0
nperseg = 512
noverlap = 256
spec_min_val = -8.0
spec_max_val = -5.0
examples_per_page = 100
cols = 10
figsize_per_panel = 1.5

ridge_min_hz = 500
ridge_max_hz = 30000
min_valid_ridge_fraction = 0.05
ridge_median_len = 3
ridge_linewidth = 0.6
ridge_color = "red"
title_fontsize = 4

# full-band clean_audio-style parameters
TV_WEIGHT = 0.08
RIDGE_SIGMAS = [1, 2]
CANNY_SIGMA = 1.2
CANNY_LOW_TH = 0.1
CANNY_HIGH_TH = 0.15
FULLBAND_OFFSET = 0.01
VOCAL_DISK_SIZE = 2
MIN_OBJ_SIZE = 20
DILATE_PIXELS = 2
MIN_COMPONENT_TIME_FRAMES = 4
MIN_COMPONENT_FREQ_SPAN_HZ = 300.0


def is_real_event_row(df: pd.DataFrame) -> pd.Series:
    start = pd.to_numeric(df[start_col], errors="coerce")
    stop = pd.to_numeric(df[stop_col], errors="coerce")
    return start.notna() & stop.notna() & (stop > start) & (start > 0)


def clean_label(label: str) -> str:
    return str(label).strip()


def find_matching_audio(csv_path: Path) -> Path | None:
    stem = csv_path.stem
    if stem.endswith("_annotations"):
        stem = stem[:-12]
    wav_path = audio_folder / f"{stem}.wav"
    if wav_path.exists():
        return wav_path
    npz_path = audio_folder / f"{stem}.npz"
    if npz_path.exists():
        return npz_path
    return None


def load_audio_file(audio_path: Path):
    suffix = audio_path.suffix.lower()
    if suffix == ".wav":
        fs, audio = wavfile.read(audio_path)
        return fs, audio
    if suffix == ".npz":
        data = np.load(audio_path, allow_pickle=True)
        audio = None
        fs = None
        for k in ["audio", "wav", "waveform", "x", "samples"]:
            if k in data.files:
                audio = data[k]
                break
        for k in ["fs", "sr", "samplerate", "sample_rate"]:
            if k in data.files:
                fs = data[k]
                break
        if audio is None or fs is None:
            raise ValueError(f"Unsupported npz layout in {audio_path.name}: {data.files}")
        if isinstance(fs, np.ndarray):
            fs = fs.item()
        return fs, audio
    raise ValueError(f"Unsupported audio type: {audio_path}")


def nanmedfilt_1d(x: np.ndarray, k: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if k <= 1:
        return x.copy()
    if k % 2 == 0:
        k += 1
    r = k // 2
    out = np.full_like(x, np.nan)
    for i in range(len(x)):
        lo = max(0, i - r)
        hi = min(len(x), i + r + 1)
        vals = x[lo:hi]
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0:
            out[i] = np.median(vals)
    return out


def make_spec_with_axes(signal: np.ndarray, fs: int):
    if signal.ndim > 1:
        signal = signal[:, 0]
    if len(signal) < nperseg:
        signal = np.pad(signal, (0, nperseg - len(signal)))
    f, t, Sxx = spectrogram(
        signal,
        fs=fs,
        nperseg=nperseg,
        noverlap=noverlap,
        scaling="spectrum",
        mode="magnitude",
    )
    logSxx = np.log(Sxx + 1e-12)
    Sxx_img = (logSxx - spec_min_val) / (spec_max_val - spec_min_val)
    Sxx_img = np.clip(Sxx_img, 0.0, 1.0)
    return f, t, logSxx, Sxx_img


def build_fullband_clean_mask(f: np.ndarray, logSxx: np.ndarray, fs: int):
    freq_mask = (f >= ridge_min_hz) & (f <= ridge_max_hz)
    f_use = f[freq_mask]
    S_use = logSxx[freq_mask, :]
    if S_use.size == 0:
        return f_use, S_use, np.zeros_like(S_use, dtype=bool)

    # normalize to 0..1 for the clean_audio-style mask
    denom = float(np.nanmax(S_use) - np.nanmin(S_use))
    if not np.isfinite(denom) or denom <= 0:
        return f_use, S_use, np.zeros_like(S_use, dtype=bool)
    S_norm = (S_use - np.nanmin(S_use)) / denom

    denoised = restoration.denoise_tv_chambolle(S_norm, weight=TV_WEIGHT)
    ridges = filters.meijering(denoised, sigmas=RIDGE_SIGMAS, black_ridges=False)
    ridge_binary = ridges > filters.threshold_otsu(ridges)
    edges = feature.canny(denoised, sigma=CANNY_SIGMA, low_threshold=CANNY_LOW_TH, high_threshold=CANNY_HIGH_TH)
    skeleton = edges | ridge_binary

    binary_full = denoised > filters.threshold_local(denoised, block_size=51, offset=FULLBAND_OFFSET)
    mask = skeleton & binary_full
    mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
    mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
    if DILATE_PIXELS > 0:
        mask = morphology.binary_dilation(mask, morphology.square(DILATE_PIXELS * 2 + 1))

    return f_use, S_use, mask


def ridge_from_best_component(f_use: np.ndarray, log_img: np.ndarray, mask: np.ndarray, t: np.ndarray):
    ridge = np.full(len(t), np.nan, dtype=float)
    if mask.size == 0 or not np.any(mask):
        return ridge, {"ridge_med_hz": np.nan, "ridge_min_hz": np.nan, "ridge_max_hz": np.nan, "fm_range_hz": np.nan, "fm_slope_hz_per_s": np.nan}

    labels, n_comp = nd_label(mask, structure=np.ones((3, 3), dtype=int))
    best_score = -np.inf
    best_comp = None

    for comp_id in range(1, n_comp + 1):
        comp = labels == comp_id
        rr, cc = np.where(comp)
        if rr.size == 0:
            continue
        time_span = int(cc.max() - cc.min() + 1)
        freq_span_hz = float(f_use[rr.max()] - f_use[rr.min()])
        if time_span < MIN_COMPONENT_TIME_FRAMES:
            continue
        mean_amp = float(np.nanmean(log_img[comp]))
        low_freq_khz = float(f_use[rr.min()] / 1000.0)
        score = mean_amp + 0.22 * time_span + 0.0030 * freq_span_hz - 0.02 * low_freq_khz
        if freq_span_hz < MIN_COMPONENT_FREQ_SPAN_HZ:
            score -= 6.0
        if score > best_score:
            best_score = score
            best_comp = comp

    if best_comp is None:
        return ridge, {"ridge_med_hz": np.nan, "ridge_min_hz": np.nan, "ridge_max_hz": np.nan, "fm_range_hz": np.nan, "fm_slope_hz_per_s": np.nan}

    for j in range(best_comp.shape[1]):
        rows = np.flatnonzero(best_comp[:, j])
        if rows.size == 0:
            continue
        weights = np.exp(log_img[rows, j] - np.nanmax(log_img[rows, j]))
        row_center = float(np.average(rows, weights=weights))
        ridge[j] = float(np.interp(row_center, np.arange(len(f_use)), f_use))

    valid = np.isfinite(ridge)
    if np.sum(valid) >= 2:
        ridge_interp = ridge.copy()
        missing = ~valid
        ridge_interp[missing] = np.interp(np.flatnonzero(missing), np.flatnonzero(valid), ridge[valid])
        ridge = ridge_interp
    ridge = nanmedfilt_1d(ridge, ridge_median_len)
    valid = np.isfinite(ridge)
    if np.sum(valid) == 0:
        return ridge, {"ridge_med_hz": np.nan, "ridge_min_hz": np.nan, "ridge_max_hz": np.nan, "fm_range_hz": np.nan, "fm_slope_hz_per_s": np.nan}

    ridge_valid = ridge[valid]
    t_valid = t[valid]
    ridge_med_hz = float(np.median(ridge_valid))
    ridge_min_hz_val = float(np.min(ridge_valid))
    ridge_max_hz_val = float(np.max(ridge_valid))
    fm_range_hz = float(ridge_max_hz_val - ridge_min_hz_val)
    if len(ridge_valid) >= 2 and np.ptp(t_valid) > 0:
        slope, _ = np.polyfit(t_valid, ridge_valid, 1)
        fm_slope_hz_per_s = float(slope)
    else:
        fm_slope_hz_per_s = np.nan
    summary = {
        "ridge_med_hz": ridge_med_hz,
        "ridge_min_hz": ridge_min_hz_val,
        "ridge_max_hz": ridge_max_hz_val,
        "fm_range_hz": fm_range_hz,
        "fm_slope_hz_per_s": fm_slope_hz_per_s,
    }
    return ridge, summary


calls_by_label = {}
csv_files = sorted(csv_folder.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in {csv_folder}")
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
    except Exception as e:
        print(f"[skip] could not read {csv_file.name}: {e}")
        continue
    needed = {label_col, start_col, stop_col}
    if not needed.issubset(df.columns):
        continue
    df = df[is_real_event_row(df)].copy()
    if len(df) == 0:
        continue
    audio_path = find_matching_audio(csv_file)
    if audio_path is None:
        continue
    df[start_col] = pd.to_numeric(df[start_col], errors="coerce")
    df[stop_col] = pd.to_numeric(df[stop_col], errors="coerce")
    for _, row in df.iterrows():
        label_name = clean_label(row[label_col])
        event = {
            "csv_file": csv_file,
            "audio_file": audio_path,
            "label": label_name,
            "start": float(row[start_col]),
            "stop": float(row[stop_col]),
        }
        calls_by_label.setdefault(label_name, []).append(event)

print("Collected labels:")
for label_name, events in sorted(calls_by_label.items()):
    print(f"  {label_name}: {len(events)} calls")

scatter_rows = []
preview_counter = 0

for label_name, events in sorted(calls_by_label.items()):
    if label_name not in target_labels:
        continue
    print(f"\nPlotting label '{label_name}' with full-band clean_audio mask, n={len(events)}")
    label_out = output_folder / label_name
    label_out.mkdir(parents=True, exist_ok=True)
    events_by_audio = {}
    for ev in events:
        events_by_audio.setdefault(ev["audio_file"], []).append(ev)
    all_specs = []
    for audio_path, audio_events in events_by_audio.items():
        try:
            fs, audio = load_audio_file(audio_path)
        except Exception as e:
            print(f"[skip audio] {audio_path.name}: {e}")
            continue
        if audio.ndim > 1:
            audio = audio[:, 0]
        n_samples = len(audio)
        for ev in audio_events:
            start = max(0, ev["start"] - buffer_sec)
            stop = ev["stop"] + buffer_sec
            s0 = int(round(start * fs))
            s1 = min(int(round(stop * fs)), n_samples)
            if s1 <= s0:
                continue
            segment = audio[s0:s1]
            try:
                f, t, logSxx, Sxx_img = make_spec_with_axes(segment, fs)
            except Exception as e:
                print(f"[skip spec] {audio_path.name}: {e}")
                continue
            f_use, log_use, clean_mask = build_fullband_clean_mask(f, logSxx, fs)
            ridge_hz, ridge_summary = ridge_from_best_component(f_use, log_use, clean_mask, t)
            valid_frac = np.mean(np.isfinite(ridge_hz)) if len(ridge_hz) > 0 else 0.0
            if valid_frac < min_valid_ridge_fraction:
                ridge_hz[:] = np.nan
                ridge_summary = {
                    "ridge_med_hz": np.nan,
                    "ridge_min_hz": np.nan,
                    "ridge_max_hz": np.nan,
                    "fm_range_hz": np.nan,
                    "fm_slope_hz_per_s": np.nan,
                }
            all_specs.append({
                "spec": Sxx_img,
                "f": f,
                "t": t,
                "ridge_hz": ridge_hz,
                "ridge_summary": ridge_summary,
                "event": ev,
            })
            preview_counter += 1
            preview_path = preview_folder / f"preview_{preview_counter:06d}.png"
            fig_preview, ax_preview = plt.subplots(figsize=(4.2, 3.0))
            ax_preview.imshow(
                Sxx_img,
                aspect="auto",
                origin="lower",
                interpolation="nearest",
                vmin=0.0,
                vmax=1.0,
                extent=[t[0], t[-1], f[0] / 1000.0, f[-1] / 1000.0],
            )
            if np.any(np.isfinite(ridge_hz)):
                valid = np.isfinite(ridge_hz)
                ax_preview.plot(t[valid], ridge_hz[valid] / 1000.0, color=ridge_color, linewidth=ridge_linewidth)
            preview_tmax = max(0.3, float(ev["stop"] - ev["start"]))
            ax_preview.set_xlim(0.0, preview_tmax)
            ax_preview.set_ylim(0.0, 60.0)
            ax_preview.set_xlabel("Time (s)")
            ax_preview.set_ylabel("Frequency (kHz)")
            fig_preview.tight_layout()
            fig_preview.savefig(preview_path, dpi=140, bbox_inches="tight")
            plt.close(fig_preview)
            scatter_rows.append({
                "label": label_name,
                "audio_name": ev["audio_file"].name,
                "ridge_med_hz": ridge_summary["ridge_med_hz"],
                "fm_range_hz": ridge_summary["fm_range_hz"],
                "duration_s": float(ev["stop"] - ev["start"]),
                "preview_path": str(preview_path),
            })
    if len(all_specs) == 0:
        print(f"[skip label] no spectrograms made for {label_name}")
        continue
    num_pages = math.ceil(len(all_specs) / examples_per_page)
    for page_idx in range(num_pages):
        start_i = page_idx * examples_per_page
        end_i = min((page_idx + 1) * examples_per_page, len(all_specs))
        page_specs = all_specs[start_i:end_i]
        n = len(page_specs)
        ncols = cols
        nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * figsize_per_panel, nrows * figsize_per_panel))
        if nrows == 1 and ncols == 1:
            axes = np.array([[axes]])
        elif nrows == 1:
            axes = np.array([axes])
        elif ncols == 1:
            axes = axes[:, np.newaxis]
        axes = axes.flatten()
        for ax, item in zip(axes, page_specs):
            Sxx = item["spec"]
            ev = item["event"]
            f = item["f"]
            t = item["t"]
            ridge_hz = item["ridge_hz"]
            ridge_summary = item["ridge_summary"]
            ax.imshow(Sxx, aspect="auto", origin="lower", interpolation="nearest", vmin=0.0, vmax=1.0, extent=[t[0], t[-1], f[0] / 1000.0, f[-1] / 1000.0])
            if np.any(np.isfinite(ridge_hz)):
                valid = np.isfinite(ridge_hz)
                ax.plot(t[valid], ridge_hz[valid] / 1000.0, color=ridge_color, linewidth=ridge_linewidth)
            ax.set_xticks([])
            ax.set_yticks([])
            ridge_med = ridge_summary["ridge_med_hz"]
            fm_range = ridge_summary["fm_range_hz"]
            if np.isfinite(ridge_med):
                title_text = f"{ev['audio_file'].name}\nridge={ridge_med/1000:.1f}k  FM={fm_range/1000:.1f}k"
            else:
                title_text = f"{ev['audio_file'].name}\nridge=nan  FM=nan"
            ax.set_title(title_text, fontsize=title_fontsize)
        for ax in axes[len(page_specs):]:
            ax.axis("off")
        fig.suptitle(f"{label_name} - page {page_idx + 1}/{num_pages} - n={len(page_specs)}", fontsize=14)
        fig.tight_layout()
        out_path = label_out / f"{label_name}_page_{page_idx + 1:03d}_fullband_cleanaudio.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
    print(f"Saved {num_pages} page(s) for label '{label_name}' to {label_out}")

if len(scatter_rows) > 0:
    scatter_df = pd.DataFrame(scatter_rows)
    scatter_df = scatter_df[np.isfinite(scatter_df["ridge_med_hz"]) & np.isfinite(scatter_df["fm_range_hz"])].copy()
    if len(scatter_df) > 0:
        color_map = {
            "mnt": "#1f77b4",
            "dense-stack": "#ff7f0e",
            "dfm-stack": "#2ca02c",
        }
        fig, ax = plt.subplots(figsize=(8, 6))
        for label_name in ["mnt", "dense-stack", "dfm-stack"]:
            sub = scatter_df[scatter_df["label"] == label_name]
            if len(sub) == 0:
                continue
            ax.scatter(
                sub["ridge_med_hz"] / 1000.0,
                sub["fm_range_hz"] / 1000.0,
                s=18,
                alpha=0.75,
                label=label_name,
                color=color_map[label_name],
            )
        ax.set_xlabel("Median ridge frequency (kHz)")
        ax.set_ylabel("FM range (kHz)")
        ax.set_title("Call summary: median frequency vs FM range")
        ax.legend(frameon=False)
        ax.grid(alpha=0.25)
        fig.tight_layout()
        scatter_path = output_folder / "summary_scatter_fullband_cleanaudio.png"
        fig.savefig(scatter_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved scatter plot to {scatter_path}")

        fig = plt.figure(figsize=(9, 7))
        ax = fig.add_subplot(111, projection="3d")
        for label_name in ["mnt", "dense-stack", "dfm-stack"]:
            sub = scatter_df[scatter_df["label"] == label_name]
            if len(sub) == 0:
                continue
            ax.scatter(
                sub["ridge_med_hz"] / 1000.0,
                sub["fm_range_hz"] / 1000.0,
                sub["duration_s"] * 1000.0,
                s=18,
                alpha=0.75,
                label=label_name,
                color=color_map[label_name],
            )
        ax.set_xlabel("Median ridge frequency (kHz)")
        ax.set_ylabel("FM range (kHz)")
        ax.set_zlabel("Duration (ms)")
        ax.set_title("Call summary: frequency, FM, duration")
        ax.legend(frameon=False)
        scatter3d_path = output_folder / "summary_scatter3d_fullband_cleanaudio.png"
        fig.tight_layout()
        fig.savefig(scatter3d_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved 3D scatter plot to {scatter3d_path}")

        try:
            import plotly.express as px
            import plotly.graph_objects as go
            import ipywidgets as widgets
            from IPython.display import display

            scatter_df = scatter_df.copy()
            scatter_df["ridge_med_khz"] = scatter_df["ridge_med_hz"] / 1000.0
            scatter_df["fm_range_khz"] = scatter_df["fm_range_hz"] / 1000.0
            scatter_df["duration_ms"] = scatter_df["duration_s"] * 1000.0

            fig3d = px.scatter_3d(
                scatter_df,
                x="ridge_med_khz",
                y="fm_range_khz",
                z="duration_ms",
                color="label",
                hover_name="audio_name",
                custom_data=["preview_path", "audio_name", "label", "ridge_med_khz", "fm_range_khz", "duration_ms"],
                hover_data={
                    "label": True,
                    "ridge_med_khz": ":.2f",
                    "fm_range_khz": ":.2f",
                    "duration_ms": ":.1f",
                },
                labels={
                    "ridge_med_khz": "Median ridge frequency (kHz)",
                    "fm_range_khz": "FM range (kHz)",
                    "duration_ms": "Duration (ms)",
                    "label": "Call type",
                },
                title="Interactive call summary: frequency, FM, duration",
                color_discrete_map=color_map,
            )
            fig3d.update_traces(marker=dict(size=4, opacity=0.75))
            fig3d.update_layout(
                scene=dict(
                    xaxis_title="Median ridge frequency (kHz)",
                    yaxis_title="FM range (kHz)",
                    zaxis_title="Duration (ms)",
                ),
                margin=dict(l=0, r=0, t=50, b=0),
                clickmode="event+select",
            )
            html_path = output_folder / "summary_scatter3d_fullband_cleanaudio.html"
            fig3d.write_html(str(html_path), include_plotlyjs="cdn")
            print(f"Saved interactive 3D scatter to {html_path}")

            figw = go.FigureWidget(fig3d)
            info_html = widgets.HTML()
            preview_out = widgets.Output()

            def _render_preview(preview_path, audio_name, call_label, ridge_med_khz, fm_range_khz, duration_ms):
                info_html.value = (
                    f"<b>{call_label}</b><br>"
                    f"{audio_name}<br>"
                    f"Median ridge: {ridge_med_khz:.2f} kHz<br>"
                    f"FM range: {fm_range_khz:.2f} kHz<br>"
                    f"Duration: {duration_ms:.1f} ms"
                )
                with preview_out:
                    preview_out.clear_output(wait=True)
                    img = plt.imread(preview_path)
                    fig_preview, ax_preview = plt.subplots(figsize=(6.8, 4.4))
                    ax_preview.imshow(img)
                    ax_preview.set_title(f"{call_label} | {audio_name}")
                    ax_preview.axis('off')
                    display(fig_preview)
                    plt.close(fig_preview)

            def _handle_click(trace, points, state):
                if not points.point_inds:
                    return
                idx = points.point_inds[0]
                preview_path, audio_name, call_label, ridge_med_khz, fm_range_khz, duration_ms = trace.customdata[idx]
                _render_preview(preview_path, audio_name, call_label, ridge_med_khz, fm_range_khz, duration_ms)

            for trace in figw.data:
                trace.on_click(_handle_click)

            if len(scatter_df):
                row0 = scatter_df.iloc[0]
                _render_preview(
                    row0['preview_path'],
                    row0['audio_name'],
                    row0['label'],
                    row0['ridge_med_khz'],
                    row0['fm_range_khz'],
                    row0['duration_ms'],
                )

            plot_box = widgets.VBox([
                widgets.HTML('<b>Interactive 3D scatter</b>'),
                figw,
            ], layout=widgets.Layout(width='58%'))
            controls = widgets.VBox([
                widgets.HTML('<b>Spectrogram preview</b>'),
                info_html,
                preview_out,
            ], layout=widgets.Layout(width='42%', padding='0 0 0 24px'))
            display(widgets.HBox([
                plot_box,
                controls,
            ], layout=widgets.Layout(align_items='flex-start', width='100%')))
        except Exception as e:
            print(f"[info] interactive 3D scatter not created: {e}")

print("\nDone with full-band clean_audio-style cell.")


\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10
Found 62 CSV files in \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10
Collected labels:
  alarm: 171 calls
  dense-stack: 126 calls
  dfm-stack: 102 calls
  high-freq: 323 calls
  mnt: 76 calls
  newborn: 13 calls
  noise: 1 calls
  warble: 208 calls

Plotting label 'dense-stack' with full-band clean_audio mask, n=126


C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2337830389.py:174: FutureWarning: `binary_opening` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.opening` instead.
  mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2337830389.py:175: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2337830389.py:177: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphol

Saved 2 page(s) for label 'dense-stack' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio\dense-stack

Plotting label 'dfm-stack' with full-band clean_audio mask, n=102


C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2337830389.py:174: FutureWarning: `binary_opening` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.opening` instead.
  mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2337830389.py:175: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2337830389.py:177: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphol

Saved 2 page(s) for label 'dfm-stack' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio\dfm-stack

Plotting label 'mnt' with full-band clean_audio mask, n=76


C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2337830389.py:174: FutureWarning: `binary_opening` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.opening` instead.
  mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2337830389.py:175: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2337830389.py:177: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphol

Saved 1 page(s) for label 'mnt' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio\mnt
Saved scatter plot to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio\summary_scatter_fullband_cleanaudio.png
Saved 3D scatter plot to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio\summary_scatter3d_fullband_cleanaudio.png
Saved interactive 3D scatter to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio\summary_scatter3d_fullband_cleanaudio.html


    'data': [{'custo…


Done with full-band clean_audio-style cell.


In [ ]:
# ============================================================
# stacks with entropy
# ============================================================

import math
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import spectrogram
from scipy.ndimage import binary_dilation, label as nd_label, median_filter
from skimage import feature, filters, morphology, restoration

if platform.system() == "Windows":
    base_das_local = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox")
else:
    base_das_local = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Pre_processing/das/models/all_vox")

csv_folder = base_das_local / "TRAINING_data_10"
audio_folder = csv_folder
output_folder = base_das_local / "TRAINING_data_10_viz_harmon_fullband_cleanaudio_entropy"
output_folder.mkdir(parents=True, exist_ok=True)
preview_folder = output_folder / "_point_previews"
preview_folder.mkdir(parents=True, exist_ok=True)
print(audio_folder)

target_labels = {"mnt", "dense-stack", "dfm-stack"}
label_col = "name"
start_col = "start_seconds"
stop_col = "stop_seconds"

buffer_sec = 0
nperseg = 512
noverlap = 256
spec_min_val = -8.0
spec_max_val = -5.0
examples_per_page = 100
cols = 10
figsize_per_panel = 1.5

ridge_min_hz = 500
ridge_max_hz = 30000
min_valid_ridge_fraction = 0.05
ridge_median_len = 3
ridge_linewidth = 0.6
ridge_color = "red"
title_fontsize = 4

# full-band clean_audio-style parameters
TV_WEIGHT = 0.08
RIDGE_SIGMAS = [1, 2]
CANNY_SIGMA = 1.2
CANNY_LOW_TH = 0.1
CANNY_HIGH_TH = 0.15
FULLBAND_OFFSET = 0.01
VOCAL_DISK_SIZE = 2
MIN_OBJ_SIZE = 20
DILATE_PIXELS = 2
MIN_COMPONENT_TIME_FRAMES = 4
MIN_COMPONENT_FREQ_SPAN_HZ = 300.0


def is_real_event_row(df: pd.DataFrame) -> pd.Series:
    start = pd.to_numeric(df[start_col], errors="coerce")
    stop = pd.to_numeric(df[stop_col], errors="coerce")
    return start.notna() & stop.notna() & (stop > start) & (start > 0)


def clean_label(label: str) -> str:
    return str(label).strip()


def find_matching_audio(csv_path: Path) -> Path | None:
    stem = csv_path.stem
    if stem.endswith("_annotations"):
        stem = stem[:-12]
    wav_path = audio_folder / f"{stem}.wav"
    if wav_path.exists():
        return wav_path
    npz_path = audio_folder / f"{stem}.npz"
    if npz_path.exists():
        return npz_path
    return None


def load_audio_file(audio_path: Path):
    suffix = audio_path.suffix.lower()
    if suffix == ".wav":
        fs, audio = wavfile.read(audio_path)
        return fs, audio
    if suffix == ".npz":
        data = np.load(audio_path, allow_pickle=True)
        audio = None
        fs = None
        for k in ["audio", "wav", "waveform", "x", "samples"]:
            if k in data.files:
                audio = data[k]
                break
        for k in ["fs", "sr", "samplerate", "sample_rate"]:
            if k in data.files:
                fs = data[k]
                break
        if audio is None or fs is None:
            raise ValueError(f"Unsupported npz layout in {audio_path.name}: {data.files}")
        if isinstance(fs, np.ndarray):
            fs = fs.item()
        return fs, audio
    raise ValueError(f"Unsupported audio type: {audio_path}")


def nanmedfilt_1d(x: np.ndarray, k: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if k <= 1:
        return x.copy()
    if k % 2 == 0:
        k += 1
    r = k // 2
    out = np.full_like(x, np.nan)
    for i in range(len(x)):
        lo = max(0, i - r)
        hi = min(len(x), i + r + 1)
        vals = x[lo:hi]
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0:
            out[i] = np.median(vals)
    return out


def make_spec_with_axes(signal: np.ndarray, fs: int):
    if signal.ndim > 1:
        signal = signal[:, 0]
    if len(signal) < nperseg:
        signal = np.pad(signal, (0, nperseg - len(signal)))
    f, t, Sxx = spectrogram(
        signal,
        fs=fs,
        nperseg=nperseg,
        noverlap=noverlap,
        scaling="spectrum",
        mode="magnitude",
    )
    logSxx = np.log(Sxx + 1e-12)
    Sxx_img = (logSxx - spec_min_val) / (spec_max_val - spec_min_val)
    Sxx_img = np.clip(Sxx_img, 0.0, 1.0)
    return f, t, logSxx, Sxx_img


def build_fullband_clean_mask(f: np.ndarray, logSxx: np.ndarray, fs: int):
    freq_mask = (f >= ridge_min_hz) & (f <= ridge_max_hz)
    f_use = f[freq_mask]
    S_use = logSxx[freq_mask, :]
    if S_use.size == 0:
        return f_use, S_use, np.zeros_like(S_use, dtype=bool)

    # normalize to 0..1 for the clean_audio-style mask
    denom = float(np.nanmax(S_use) - np.nanmin(S_use))
    if not np.isfinite(denom) or denom <= 0:
        return f_use, S_use, np.zeros_like(S_use, dtype=bool)
    S_norm = (S_use - np.nanmin(S_use)) / denom

    denoised = restoration.denoise_tv_chambolle(S_norm, weight=TV_WEIGHT)
    ridges = filters.meijering(denoised, sigmas=RIDGE_SIGMAS, black_ridges=False)
    ridge_binary = ridges > filters.threshold_otsu(ridges)
    edges = feature.canny(denoised, sigma=CANNY_SIGMA, low_threshold=CANNY_LOW_TH, high_threshold=CANNY_HIGH_TH)
    skeleton = edges | ridge_binary

    binary_full = denoised > filters.threshold_local(denoised, block_size=51, offset=FULLBAND_OFFSET)
    mask = skeleton & binary_full
    mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
    mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
    if DILATE_PIXELS > 0:
        mask = morphology.binary_dilation(mask, morphology.square(DILATE_PIXELS * 2 + 1))

    return f_use, S_use, mask


def spectral_entropy_from_logspec(log_img: np.ndarray):
    if log_img.size == 0:
        return np.nan
    power = np.power(10.0, log_img)
    power = np.clip(power, 0.0, None)
    col_sums = np.sum(power, axis=0, keepdims=True)
    valid = np.squeeze(col_sums > 0, axis=0)
    if not np.any(valid):
        return np.nan
    p = power[:, valid] / col_sums[:, valid]
    p = np.clip(p, 1e-12, None)
    ent = -np.sum(p * np.log2(p), axis=0) / np.log2(power.shape[0])
    if ent.size == 0:
        return np.nan
    return float(np.mean(ent))


def ridge_from_best_component(f_use: np.ndarray, log_img: np.ndarray, mask: np.ndarray, t: np.ndarray):
    ridge = np.full(len(t), np.nan, dtype=float)
    if mask.size == 0 or not np.any(mask):
        return ridge, {"ridge_med_hz": np.nan, "ridge_min_hz": np.nan, "ridge_max_hz": np.nan, "fm_range_hz": np.nan, "fm_slope_hz_per_s": np.nan}

    labels, n_comp = nd_label(mask, structure=np.ones((3, 3), dtype=int))
    best_score = -np.inf
    best_comp = None

    for comp_id in range(1, n_comp + 1):
        comp = labels == comp_id
        rr, cc = np.where(comp)
        if rr.size == 0:
            continue
        time_span = int(cc.max() - cc.min() + 1)
        freq_span_hz = float(f_use[rr.max()] - f_use[rr.min()])
        if time_span < MIN_COMPONENT_TIME_FRAMES:
            continue
        mean_amp = float(np.nanmean(log_img[comp]))
        low_freq_khz = float(f_use[rr.min()] / 1000.0)
        score = mean_amp + 0.22 * time_span + 0.0030 * freq_span_hz - 0.02 * low_freq_khz
        if freq_span_hz < MIN_COMPONENT_FREQ_SPAN_HZ:
            score -= 6.0
        if score > best_score:
            best_score = score
            best_comp = comp

    if best_comp is None:
        return ridge, {"ridge_med_hz": np.nan, "ridge_min_hz": np.nan, "ridge_max_hz": np.nan, "fm_range_hz": np.nan, "fm_slope_hz_per_s": np.nan}

    for j in range(best_comp.shape[1]):
        rows = np.flatnonzero(best_comp[:, j])
        if rows.size == 0:
            continue
        weights = np.exp(log_img[rows, j] - np.nanmax(log_img[rows, j]))
        row_center = float(np.average(rows, weights=weights))
        ridge[j] = float(np.interp(row_center, np.arange(len(f_use)), f_use))

    valid = np.isfinite(ridge)
    if np.sum(valid) >= 2:
        ridge_interp = ridge.copy()
        missing = ~valid
        ridge_interp[missing] = np.interp(np.flatnonzero(missing), np.flatnonzero(valid), ridge[valid])
        ridge = ridge_interp
    ridge = nanmedfilt_1d(ridge, ridge_median_len)
    valid = np.isfinite(ridge)
    if np.sum(valid) == 0:
        return ridge, {"ridge_med_hz": np.nan, "ridge_min_hz": np.nan, "ridge_max_hz": np.nan, "fm_range_hz": np.nan, "fm_slope_hz_per_s": np.nan}

    ridge_valid = ridge[valid]
    t_valid = t[valid]
    ridge_med_hz = float(np.median(ridge_valid))
    ridge_min_hz_val = float(np.min(ridge_valid))
    ridge_max_hz_val = float(np.max(ridge_valid))
    fm_range_hz = float(ridge_max_hz_val - ridge_min_hz_val)
    if len(ridge_valid) >= 2 and np.ptp(t_valid) > 0:
        slope, _ = np.polyfit(t_valid, ridge_valid, 1)
        fm_slope_hz_per_s = float(slope)
    else:
        fm_slope_hz_per_s = np.nan
    summary = {
        "ridge_med_hz": ridge_med_hz,
        "ridge_min_hz": ridge_min_hz_val,
        "ridge_max_hz": ridge_max_hz_val,
        "fm_range_hz": fm_range_hz,
        "fm_slope_hz_per_s": fm_slope_hz_per_s,
    }
    return ridge, summary


calls_by_label = {}
csv_files = sorted(csv_folder.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in {csv_folder}")
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
    except Exception as e:
        print(f"[skip] could not read {csv_file.name}: {e}")
        continue
    needed = {label_col, start_col, stop_col}
    if not needed.issubset(df.columns):
        continue
    df = df[is_real_event_row(df)].copy()
    if len(df) == 0:
        continue
    audio_path = find_matching_audio(csv_file)
    if audio_path is None:
        continue
    df[start_col] = pd.to_numeric(df[start_col], errors="coerce")
    df[stop_col] = pd.to_numeric(df[stop_col], errors="coerce")
    for _, row in df.iterrows():
        label_name = clean_label(row[label_col])
        event = {
            "csv_file": csv_file,
            "audio_file": audio_path,
            "label": label_name,
            "start": float(row[start_col]),
            "stop": float(row[stop_col]),
        }
        calls_by_label.setdefault(label_name, []).append(event)

print("Collected labels:")
for label_name, events in sorted(calls_by_label.items()):
    print(f"  {label_name}: {len(events)} calls")

scatter_rows = []
preview_counter = 0

for label_name, events in sorted(calls_by_label.items()):
    if label_name not in target_labels:
        continue
    print(f"\nPlotting label '{label_name}' with full-band clean_audio mask, n={len(events)}")
    label_out = output_folder / label_name
    label_out.mkdir(parents=True, exist_ok=True)
    events_by_audio = {}
    for ev in events:
        events_by_audio.setdefault(ev["audio_file"], []).append(ev)
    all_specs = []
    for audio_path, audio_events in events_by_audio.items():
        try:
            fs, audio = load_audio_file(audio_path)
        except Exception as e:
            print(f"[skip audio] {audio_path.name}: {e}")
            continue
        if audio.ndim > 1:
            audio = audio[:, 0]
        n_samples = len(audio)
        for ev in audio_events:
            start = max(0, ev["start"] - buffer_sec)
            stop = ev["stop"] + buffer_sec
            s0 = int(round(start * fs))
            s1 = min(int(round(stop * fs)), n_samples)
            if s1 <= s0:
                continue
            segment = audio[s0:s1]
            try:
                f, t, logSxx, Sxx_img = make_spec_with_axes(segment, fs)
            except Exception as e:
                print(f"[skip spec] {audio_path.name}: {e}")
                continue
            f_use, log_use, clean_mask = build_fullband_clean_mask(f, logSxx, fs)
            ridge_hz, ridge_summary = ridge_from_best_component(f_use, log_use, clean_mask, t)
            valid_frac = np.mean(np.isfinite(ridge_hz)) if len(ridge_hz) > 0 else 0.0
            if valid_frac < min_valid_ridge_fraction:
                ridge_hz[:] = np.nan
                ridge_summary = {
                    "ridge_med_hz": np.nan,
                    "ridge_min_hz": np.nan,
                    "ridge_max_hz": np.nan,
                    "fm_range_hz": np.nan,
                    "fm_slope_hz_per_s": np.nan,
                }
            all_specs.append({
                "spec": Sxx_img,
                "f": f,
                "t": t,
                "ridge_hz": ridge_hz,
                "ridge_summary": ridge_summary,
                "event": ev,
            })
            preview_counter += 1
            preview_path = preview_folder / f"preview_{preview_counter:06d}.png"
            fig_preview, ax_preview = plt.subplots(figsize=(4.2, 3.0))
            ax_preview.imshow(
                Sxx_img,
                aspect="auto",
                origin="lower",
                interpolation="nearest",
                vmin=0.0,
                vmax=1.0,
                extent=[t[0], t[-1], f[0] / 1000.0, f[-1] / 1000.0],
            )
            if np.any(np.isfinite(ridge_hz)):
                valid = np.isfinite(ridge_hz)
                ax_preview.plot(t[valid], ridge_hz[valid] / 1000.0, color=ridge_color, linewidth=ridge_linewidth)
            preview_tmax = max(0.3, float(ev["stop"] - ev["start"]))
            ax_preview.set_xlim(0.0, preview_tmax)
            ax_preview.set_ylim(0.0, 60.0)
            ax_preview.set_xlabel("Time (s)")
            ax_preview.set_ylabel("Frequency (kHz)")
            fig_preview.tight_layout()
            fig_preview.savefig(preview_path, dpi=140, bbox_inches="tight")
            plt.close(fig_preview)
            spec_entropy = spectral_entropy_from_logspec(log_use)
            scatter_rows.append({
                "label": label_name,
                "audio_name": ev["audio_file"].name,
                "ridge_med_hz": ridge_summary["ridge_med_hz"],
                "fm_range_hz": ridge_summary["fm_range_hz"],
                "spectral_entropy": spec_entropy,
                "preview_path": str(preview_path),
            })
    if len(all_specs) == 0:
        print(f"[skip label] no spectrograms made for {label_name}")
        continue
    num_pages = math.ceil(len(all_specs) / examples_per_page)
    for page_idx in range(num_pages):
        start_i = page_idx * examples_per_page
        end_i = min((page_idx + 1) * examples_per_page, len(all_specs))
        page_specs = all_specs[start_i:end_i]
        n = len(page_specs)
        ncols = cols
        nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * figsize_per_panel, nrows * figsize_per_panel))
        if nrows == 1 and ncols == 1:
            axes = np.array([[axes]])
        elif nrows == 1:
            axes = np.array([axes])
        elif ncols == 1:
            axes = axes[:, np.newaxis]
        axes = axes.flatten()
        for ax, item in zip(axes, page_specs):
            Sxx = item["spec"]
            ev = item["event"]
            f = item["f"]
            t = item["t"]
            ridge_hz = item["ridge_hz"]
            ridge_summary = item["ridge_summary"]
            ax.imshow(Sxx, aspect="auto", origin="lower", interpolation="nearest", vmin=0.0, vmax=1.0, extent=[t[0], t[-1], f[0] / 1000.0, f[-1] / 1000.0])
            if np.any(np.isfinite(ridge_hz)):
                valid = np.isfinite(ridge_hz)
                ax.plot(t[valid], ridge_hz[valid] / 1000.0, color=ridge_color, linewidth=ridge_linewidth)
            ax.set_xticks([])
            ax.set_yticks([])
            ridge_med = ridge_summary["ridge_med_hz"]
            fm_range = ridge_summary["fm_range_hz"]
            if np.isfinite(ridge_med):
                title_text = f"{ev['audio_file'].name}\nridge={ridge_med/1000:.1f}k  FM={fm_range/1000:.1f}k"
            else:
                title_text = f"{ev['audio_file'].name}\nridge=nan  FM=nan"
            ax.set_title(title_text, fontsize=title_fontsize)
        for ax in axes[len(page_specs):]:
            ax.axis("off")
        fig.suptitle(f"{label_name} - page {page_idx + 1}/{num_pages} - n={len(page_specs)}", fontsize=14)
        fig.tight_layout()
        out_path = label_out / f"{label_name}_page_{page_idx + 1:03d}_fullband_cleanaudio.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
    print(f"Saved {num_pages} page(s) for label '{label_name}' to {label_out}")

if len(scatter_rows) > 0:
    scatter_df = pd.DataFrame(scatter_rows)
    scatter_df = scatter_df[np.isfinite(scatter_df["ridge_med_hz"]) & np.isfinite(scatter_df["fm_range_hz"])].copy()
    if len(scatter_df) > 0:
        color_map = {
            "mnt": "#1f77b4",
            "dense-stack": "#ff7f0e",
            "dfm-stack": "#2ca02c",
        }
        fig, ax = plt.subplots(figsize=(8, 6))
        for label_name in ["mnt", "dense-stack", "dfm-stack"]:
            sub = scatter_df[scatter_df["label"] == label_name]
            if len(sub) == 0:
                continue
            ax.scatter(
                sub["ridge_med_hz"] / 1000.0,
                sub["fm_range_hz"] / 1000.0,
                s=18,
                alpha=0.75,
                label=label_name,
                color=color_map[label_name],
            )
        ax.set_xlabel("Median ridge frequency (kHz)")
        ax.set_ylabel("FM range (kHz)")
        ax.set_title("Call summary: median frequency vs FM range")
        ax.legend(frameon=False)
        ax.grid(alpha=0.25)
        fig.tight_layout()
        scatter_path = output_folder / "summary_scatter_fullband_cleanaudio_entropy.png"
        fig.savefig(scatter_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved scatter plot to {scatter_path}")

        fig = plt.figure(figsize=(9, 7))
        ax = fig.add_subplot(111, projection="3d")
        for label_name in ["mnt", "dense-stack", "dfm-stack"]:
            sub = scatter_df[scatter_df["label"] == label_name]
            if len(sub) == 0:
                continue
            ax.scatter(
                sub["ridge_med_hz"] / 1000.0,
                sub["fm_range_hz"] / 1000.0,
                sub["spectral_entropy"],
                s=18,
                alpha=0.75,
                label=label_name,
                color=color_map[label_name],
            )
        ax.set_xlabel("Median ridge frequency (kHz)")
        ax.set_ylabel("FM range (kHz)")
        ax.set_zlabel("Spectral entropy")
        ax.set_title("Call summary: frequency, FM, spectral entropy")
        ax.legend(frameon=False)
        scatter3d_path = output_folder / "summary_scatter3d_fullband_cleanaudio_entropy.png"
        fig.tight_layout()
        fig.savefig(scatter3d_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved 3D scatter plot to {scatter3d_path}")

        try:
            import plotly.express as px
            import plotly.graph_objects as go
            import ipywidgets as widgets
            from IPython.display import display

            scatter_df = scatter_df.copy()
            scatter_df["ridge_med_khz"] = scatter_df["ridge_med_hz"] / 1000.0
            scatter_df["fm_range_khz"] = scatter_df["fm_range_hz"] / 1000.0

            fig3d = px.scatter_3d(
                scatter_df,
                x="ridge_med_khz",
                y="fm_range_khz",
                z="spectral_entropy",
                color="label",
                hover_name="audio_name",
                custom_data=["preview_path", "audio_name", "label", "ridge_med_khz", "fm_range_khz", "spectral_entropy"],
                hover_data={
                    "label": True,
                    "ridge_med_khz": ":.2f",
                    "fm_range_khz": ":.2f",
                    "spectral_entropy": ":.3f",
                },
                labels={
                    "ridge_med_khz": "Median ridge frequency (kHz)",
                    "fm_range_khz": "FM range (kHz)",
                    "spectral_entropy": "Spectral entropy",
                    "label": "Call type",
                },
                title="Interactive call summary: frequency, FM, spectral entropy",
                color_discrete_map=color_map,
            )
            fig3d.update_traces(marker=dict(size=4, opacity=0.75))
            fig3d.update_layout(
                scene=dict(
                    xaxis_title="Median ridge frequency (kHz)",
                    yaxis_title="FM range (kHz)",
                    zaxis_title="Spectral entropy",
                ),
                margin=dict(l=0, r=0, t=50, b=0),
                clickmode="event+select",
            )
            html_path = output_folder / "summary_scatter3d_fullband_cleanaudio_entropy.html"
            fig3d.write_html(str(html_path), include_plotlyjs="cdn")
            print(f"Saved interactive 3D scatter to {html_path}")

            figw = go.FigureWidget(fig3d)
            info_html = widgets.HTML()
            preview_out = widgets.Output()

            def _render_preview(preview_path, audio_name, call_label, ridge_med_khz, fm_range_khz, spectral_entropy):
                info_html.value = (
                    f"<b>{call_label}</b><br>"
                    f"{audio_name}<br>"
                    f"Median ridge: {ridge_med_khz:.2f} kHz<br>"
                    f"FM range: {fm_range_khz:.2f} kHz<br>"
                    f"Spectral entropy: {spectral_entropy:.3f}"
                )
                with preview_out:
                    preview_out.clear_output(wait=True)
                    img = plt.imread(preview_path)
                    fig_preview, ax_preview = plt.subplots(figsize=(6.8, 4.4))
                    ax_preview.imshow(img)
                    ax_preview.set_title(f"{call_label} | {audio_name}")
                    ax_preview.axis('off')
                    display(fig_preview)
                    plt.close(fig_preview)

            def _handle_click(trace, points, state):
                if not points.point_inds:
                    return
                idx = points.point_inds[0]
                preview_path, audio_name, call_label, ridge_med_khz, fm_range_khz, spectral_entropy = trace.customdata[idx]
                _render_preview(preview_path, audio_name, call_label, ridge_med_khz, fm_range_khz, spectral_entropy)

            for trace in figw.data:
                trace.on_click(_handle_click)

            if len(scatter_df):
                row0 = scatter_df.iloc[0]
                _render_preview(
                    row0['preview_path'],
                    row0['audio_name'],
                    row0['label'],
                    row0['ridge_med_khz'],
                    row0['fm_range_khz'],
                    row0['spectral_entropy'],
                )

            plot_box = widgets.VBox([
                widgets.HTML('<b>Interactive 3D scatter</b>'),
                figw,
            ], layout=widgets.Layout(width='58%'))
            controls = widgets.VBox([
                widgets.HTML('<b>Spectrogram preview</b>'),
                info_html,
                preview_out,
            ], layout=widgets.Layout(width='42%', padding='0 0 0 24px'))
            display(widgets.HBox([
                plot_box,
                controls,
            ], layout=widgets.Layout(align_items='flex-start', width='100%')))
        except Exception as e:
            print(f"[info] interactive 3D scatter not created: {e}")

print("\nDone with full-band clean_audio-style cell.")


\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10
Found 62 CSV files in \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10
Collected labels:
  alarm: 171 calls
  dense-stack: 126 calls
  dfm-stack: 102 calls
  high-freq: 323 calls
  mnt: 76 calls
  newborn: 13 calls
  noise: 1 calls
  warble: 208 calls

Plotting label 'dense-stack' with full-band clean_audio mask, n=126


C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2059198301.py:172: FutureWarning: `binary_opening` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.opening` instead.
  mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2059198301.py:173: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2059198301.py:175: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphol

Saved 2 page(s) for label 'dense-stack' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_entropy\dense-stack

Plotting label 'dfm-stack' with full-band clean_audio mask, n=102


C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2059198301.py:172: FutureWarning: `binary_opening` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.opening` instead.
  mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2059198301.py:173: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2059198301.py:175: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphol

Saved 2 page(s) for label 'dfm-stack' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_entropy\dfm-stack

Plotting label 'mnt' with full-band clean_audio mask, n=76


C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2059198301.py:172: FutureWarning: `binary_opening` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.opening` instead.
  mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2059198301.py:173: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\2059198301.py:175: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphol

Saved 1 page(s) for label 'mnt' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_entropy\mnt
Saved scatter plot to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_entropy\summary_scatter_fullband_cleanaudio_entropy.png
Saved 3D scatter plot to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_entropy\summary_scatter3d_fullband_cleanaudio_entropy.png
Saved interactive 3D scatter to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_entropy\summary_scatter3d_fullband_cleanaudio_entropy.html


    'data': [{'custo…


Done with full-band clean_audio-style cell.


In [ ]:
# ============================================================
# high freq calls
# ============================================================

import math
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import spectrogram
from scipy.ndimage import binary_dilation, label as nd_label, median_filter
from skimage import feature, filters, morphology, restoration

if platform.system() == "Windows":
    base_das_local = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox")
else:
    base_das_local = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Pre_processing/das/models/all_vox")

csv_folder = base_das_local / "TRAINING_data_10"
audio_folder = csv_folder
output_folder = base_das_local / "TRAINING_data_10_viz_harmon_fullband_cleanaudio_allcalls"
output_folder.mkdir(parents=True, exist_ok=True)
preview_folder = output_folder / "_point_previews"
preview_folder.mkdir(parents=True, exist_ok=True)
print(audio_folder)

target_labels = {"high-freq", "warble","alarm"} #None  
label_col = "name"
start_col = "start_seconds"
stop_col = "stop_seconds"

buffer_sec = 0
nperseg = 512
noverlap = 256
spec_min_val = -8.0
spec_max_val = -5.0
examples_per_page = 100
cols = 10
figsize_per_panel = 1.5

ridge_min_hz = 500
ridge_max_hz = 30000
min_valid_ridge_fraction = 0.05
ridge_median_len = 3
ridge_linewidth = 0.6
ridge_color = "red"
title_fontsize = 4

# full-band clean_audio-style parameters
TV_WEIGHT = 0.08
RIDGE_SIGMAS = [1, 2]
CANNY_SIGMA = 1.2
CANNY_LOW_TH = 0.1
CANNY_HIGH_TH = 0.15
FULLBAND_OFFSET = 0.01
VOCAL_DISK_SIZE = 2
MIN_OBJ_SIZE = 20
DILATE_PIXELS = 2
MIN_COMPONENT_TIME_FRAMES = 4
MIN_COMPONENT_FREQ_SPAN_HZ = 300.0


def is_real_event_row(df: pd.DataFrame) -> pd.Series:
    start = pd.to_numeric(df[start_col], errors="coerce")
    stop = pd.to_numeric(df[stop_col], errors="coerce")
    return start.notna() & stop.notna() & (stop > start) & (start > 0)


def clean_label(label: str) -> str:
    return str(label).strip()


def find_matching_audio(csv_path: Path) -> Path | None:
    stem = csv_path.stem
    if stem.endswith("_annotations"):
        stem = stem[:-12]
    wav_path = audio_folder / f"{stem}.wav"
    if wav_path.exists():
        return wav_path
    npz_path = audio_folder / f"{stem}.npz"
    if npz_path.exists():
        return npz_path
    return None


def load_audio_file(audio_path: Path):
    suffix = audio_path.suffix.lower()
    if suffix == ".wav":
        fs, audio = wavfile.read(audio_path)
        return fs, audio
    if suffix == ".npz":
        data = np.load(audio_path, allow_pickle=True)
        audio = None
        fs = None
        for k in ["audio", "wav", "waveform", "x", "samples"]:
            if k in data.files:
                audio = data[k]
                break
        for k in ["fs", "sr", "samplerate", "sample_rate"]:
            if k in data.files:
                fs = data[k]
                break
        if audio is None or fs is None:
            raise ValueError(f"Unsupported npz layout in {audio_path.name}: {data.files}")
        if isinstance(fs, np.ndarray):
            fs = fs.item()
        return fs, audio
    raise ValueError(f"Unsupported audio type: {audio_path}")


def nanmedfilt_1d(x: np.ndarray, k: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if k <= 1:
        return x.copy()
    if k % 2 == 0:
        k += 1
    r = k // 2
    out = np.full_like(x, np.nan)
    for i in range(len(x)):
        lo = max(0, i - r)
        hi = min(len(x), i + r + 1)
        vals = x[lo:hi]
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0:
            out[i] = np.median(vals)
    return out


def make_spec_with_axes(signal: np.ndarray, fs: int):
    if signal.ndim > 1:
        signal = signal[:, 0]
    if len(signal) < nperseg:
        signal = np.pad(signal, (0, nperseg - len(signal)))
    f, t, Sxx = spectrogram(
        signal,
        fs=fs,
        nperseg=nperseg,
        noverlap=noverlap,
        scaling="spectrum",
        mode="magnitude",
    )
    logSxx = np.log(Sxx + 1e-12)
    Sxx_img = (logSxx - spec_min_val) / (spec_max_val - spec_min_val)
    Sxx_img = np.clip(Sxx_img, 0.0, 1.0)
    return f, t, logSxx, Sxx_img


def build_fullband_clean_mask(f: np.ndarray, logSxx: np.ndarray, fs: int):
    freq_mask = (f >= ridge_min_hz) & (f <= ridge_max_hz)
    f_use = f[freq_mask]
    S_use = logSxx[freq_mask, :]
    if S_use.size == 0:
        return f_use, S_use, np.zeros_like(S_use, dtype=bool)

    # normalize to 0..1 for the clean_audio-style mask
    denom = float(np.nanmax(S_use) - np.nanmin(S_use))
    if not np.isfinite(denom) or denom <= 0:
        return f_use, S_use, np.zeros_like(S_use, dtype=bool)
    S_norm = (S_use - np.nanmin(S_use)) / denom

    denoised = restoration.denoise_tv_chambolle(S_norm, weight=TV_WEIGHT)
    ridges = filters.meijering(denoised, sigmas=RIDGE_SIGMAS, black_ridges=False)
    ridge_binary = ridges > filters.threshold_otsu(ridges)
    edges = feature.canny(denoised, sigma=CANNY_SIGMA, low_threshold=CANNY_LOW_TH, high_threshold=CANNY_HIGH_TH)
    skeleton = edges | ridge_binary

    binary_full = denoised > filters.threshold_local(denoised, block_size=51, offset=FULLBAND_OFFSET)
    mask = skeleton & binary_full
    mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
    mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
    if DILATE_PIXELS > 0:
        mask = morphology.binary_dilation(mask, morphology.square(DILATE_PIXELS * 2 + 1))

    return f_use, S_use, mask


def ridge_from_best_component(f_use: np.ndarray, log_img: np.ndarray, mask: np.ndarray, t: np.ndarray):
    ridge = np.full(len(t), np.nan, dtype=float)
    if mask.size == 0 or not np.any(mask):
        return ridge, {"ridge_med_hz": np.nan, "ridge_min_hz": np.nan, "ridge_max_hz": np.nan, "fm_range_hz": np.nan, "fm_slope_hz_per_s": np.nan}

    labels, n_comp = nd_label(mask, structure=np.ones((3, 3), dtype=int))
    best_score = -np.inf
    best_comp = None

    for comp_id in range(1, n_comp + 1):
        comp = labels == comp_id
        rr, cc = np.where(comp)
        if rr.size == 0:
            continue
        time_span = int(cc.max() - cc.min() + 1)
        freq_span_hz = float(f_use[rr.max()] - f_use[rr.min()])
        if time_span < MIN_COMPONENT_TIME_FRAMES:
            continue
        mean_amp = float(np.nanmean(log_img[comp]))
        low_freq_khz = float(f_use[rr.min()] / 1000.0)
        score = mean_amp + 0.22 * time_span + 0.0030 * freq_span_hz - 0.02 * low_freq_khz
        if freq_span_hz < MIN_COMPONENT_FREQ_SPAN_HZ:
            score -= 6.0
        if score > best_score:
            best_score = score
            best_comp = comp

    if best_comp is None:
        return ridge, {"ridge_med_hz": np.nan, "ridge_min_hz": np.nan, "ridge_max_hz": np.nan, "fm_range_hz": np.nan, "fm_slope_hz_per_s": np.nan}

    for j in range(best_comp.shape[1]):
        rows = np.flatnonzero(best_comp[:, j])
        if rows.size == 0:
            continue
        weights = np.exp(log_img[rows, j] - np.nanmax(log_img[rows, j]))
        row_center = float(np.average(rows, weights=weights))
        ridge[j] = float(np.interp(row_center, np.arange(len(f_use)), f_use))

    valid = np.isfinite(ridge)
    if np.sum(valid) >= 2:
        ridge_interp = ridge.copy()
        missing = ~valid
        ridge_interp[missing] = np.interp(np.flatnonzero(missing), np.flatnonzero(valid), ridge[valid])
        ridge = ridge_interp
    ridge = nanmedfilt_1d(ridge, ridge_median_len)
    valid = np.isfinite(ridge)
    if np.sum(valid) == 0:
        return ridge, {"ridge_med_hz": np.nan, "ridge_min_hz": np.nan, "ridge_max_hz": np.nan, "fm_range_hz": np.nan, "fm_slope_hz_per_s": np.nan}

    ridge_valid = ridge[valid]
    t_valid = t[valid]
    ridge_med_hz = float(np.median(ridge_valid))
    ridge_min_hz_val = float(np.min(ridge_valid))
    ridge_max_hz_val = float(np.max(ridge_valid))
    fm_range_hz = float(ridge_max_hz_val - ridge_min_hz_val)
    if len(ridge_valid) >= 2 and np.ptp(t_valid) > 0:
        slope, _ = np.polyfit(t_valid, ridge_valid, 1)
        fm_slope_hz_per_s = float(slope)
    else:
        fm_slope_hz_per_s = np.nan
    summary = {
        "ridge_med_hz": ridge_med_hz,
        "ridge_min_hz": ridge_min_hz_val,
        "ridge_max_hz": ridge_max_hz_val,
        "fm_range_hz": fm_range_hz,
        "fm_slope_hz_per_s": fm_slope_hz_per_s,
    }
    return ridge, summary


calls_by_label = {}
csv_files = sorted(csv_folder.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in {csv_folder}")
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
    except Exception as e:
        print(f"[skip] could not read {csv_file.name}: {e}")
        continue
    needed = {label_col, start_col, stop_col}
    if not needed.issubset(df.columns):
        continue
    df = df[is_real_event_row(df)].copy()
    if len(df) == 0:
        continue
    audio_path = find_matching_audio(csv_file)
    if audio_path is None:
        continue
    df[start_col] = pd.to_numeric(df[start_col], errors="coerce")
    df[stop_col] = pd.to_numeric(df[stop_col], errors="coerce")
    for _, row in df.iterrows():
        label_name = clean_label(row[label_col])
        event = {
            "csv_file": csv_file,
            "audio_file": audio_path,
            "label": label_name,
            "start": float(row[start_col]),
            "stop": float(row[stop_col]),
        }
        calls_by_label.setdefault(label_name, []).append(event)

print("Collected labels:")
for label_name, events in sorted(calls_by_label.items()):
    print(f"  {label_name}: {len(events)} calls")

scatter_rows = []
preview_counter = 0

for label_name, events in sorted(calls_by_label.items()):
    if (target_labels is not None) and (label_name not in target_labels):
        continue
    print(f"\nPlotting label '{label_name}' with all-call full-band clean_audio mask, n={len(events)}")
    label_out = output_folder / label_name
    label_out.mkdir(parents=True, exist_ok=True)
    events_by_audio = {}
    for ev in events:
        events_by_audio.setdefault(ev["audio_file"], []).append(ev)
    all_specs = []
    for audio_path, audio_events in events_by_audio.items():
        try:
            fs, audio = load_audio_file(audio_path)
        except Exception as e:
            print(f"[skip audio] {audio_path.name}: {e}")
            continue
        if audio.ndim > 1:
            audio = audio[:, 0]
        n_samples = len(audio)
        for ev in audio_events:
            start = max(0, ev["start"] - buffer_sec)
            stop = ev["stop"] + buffer_sec
            s0 = int(round(start * fs))
            s1 = min(int(round(stop * fs)), n_samples)
            if s1 <= s0:
                continue
            segment = audio[s0:s1]
            try:
                f, t, logSxx, Sxx_img = make_spec_with_axes(segment, fs)
            except Exception as e:
                print(f"[skip spec] {audio_path.name}: {e}")
                continue
            f_use, log_use, clean_mask = build_fullband_clean_mask(f, logSxx, fs)
            ridge_hz, ridge_summary = ridge_from_best_component(f_use, log_use, clean_mask, t)
            valid_frac = np.mean(np.isfinite(ridge_hz)) if len(ridge_hz) > 0 else 0.0
            if valid_frac < min_valid_ridge_fraction:
                ridge_hz[:] = np.nan
                ridge_summary = {
                    "ridge_med_hz": np.nan,
                    "ridge_min_hz": np.nan,
                    "ridge_max_hz": np.nan,
                    "fm_range_hz": np.nan,
                    "fm_slope_hz_per_s": np.nan,
                }
            all_specs.append({
                "spec": Sxx_img,
                "f": f,
                "t": t,
                "ridge_hz": ridge_hz,
                "ridge_summary": ridge_summary,
                "event": ev,
            })
            preview_counter += 1
            preview_path = preview_folder / f"preview_{preview_counter:06d}.png"
            fig_preview, ax_preview = plt.subplots(figsize=(4.2, 3.0))
            ax_preview.imshow(
                Sxx_img,
                aspect="auto",
                origin="lower",
                interpolation="nearest",
                vmin=0.0,
                vmax=1.0,
                extent=[t[0], t[-1], f[0] / 1000.0, f[-1] / 1000.0],
            )
            if np.any(np.isfinite(ridge_hz)):
                valid = np.isfinite(ridge_hz)
                ax_preview.plot(t[valid], ridge_hz[valid] / 1000.0, color=ridge_color, linewidth=ridge_linewidth)
            preview_tmax = max(0.3, float(ev["stop"] - ev["start"]))
            ax_preview.set_xlim(0.0, preview_tmax)
            ax_preview.set_ylim(0.0, 60.0)
            ax_preview.set_xlabel("Time (s)")
            ax_preview.set_ylabel("Frequency (kHz)")
            fig_preview.tight_layout()
            fig_preview.savefig(preview_path, dpi=140, bbox_inches="tight")
            plt.close(fig_preview)
            scatter_rows.append({
                "label": label_name,
                "audio_name": ev["audio_file"].name,
                "ridge_med_hz": ridge_summary["ridge_med_hz"],
                "fm_range_hz": ridge_summary["fm_range_hz"],
                "duration_s": float(ev["stop"] - ev["start"]),
                "preview_path": str(preview_path),
            })
    if len(all_specs) == 0:
        print(f"[skip label] no spectrograms made for {label_name}")
        continue
    num_pages = math.ceil(len(all_specs) / examples_per_page)
    for page_idx in range(num_pages):
        start_i = page_idx * examples_per_page
        end_i = min((page_idx + 1) * examples_per_page, len(all_specs))
        page_specs = all_specs[start_i:end_i]
        n = len(page_specs)
        ncols = cols
        nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * figsize_per_panel, nrows * figsize_per_panel))
        if nrows == 1 and ncols == 1:
            axes = np.array([[axes]])
        elif nrows == 1:
            axes = np.array([axes])
        elif ncols == 1:
            axes = axes[:, np.newaxis]
        axes = axes.flatten()
        for ax, item in zip(axes, page_specs):
            Sxx = item["spec"]
            ev = item["event"]
            f = item["f"]
            t = item["t"]
            ridge_hz = item["ridge_hz"]
            ridge_summary = item["ridge_summary"]
            ax.imshow(Sxx, aspect="auto", origin="lower", interpolation="nearest", vmin=0.0, vmax=1.0, extent=[t[0], t[-1], f[0] / 1000.0, f[-1] / 1000.0])
            if np.any(np.isfinite(ridge_hz)):
                valid = np.isfinite(ridge_hz)
                ax.plot(t[valid], ridge_hz[valid] / 1000.0, color=ridge_color, linewidth=ridge_linewidth)
            ax.set_xticks([])
            ax.set_yticks([])
            ridge_med = ridge_summary["ridge_med_hz"]
            fm_range = ridge_summary["fm_range_hz"]
            if np.isfinite(ridge_med):
                title_text = f"{ev['audio_file'].name}\nridge={ridge_med/1000:.1f}k  FM={fm_range/1000:.1f}k"
            else:
                title_text = f"{ev['audio_file'].name}\nridge=nan  FM=nan"
            ax.set_title(title_text, fontsize=title_fontsize)
        for ax in axes[len(page_specs):]:
            ax.axis("off")
        fig.suptitle(f"{label_name} - page {page_idx + 1}/{num_pages} - n={len(page_specs)}", fontsize=14)
        fig.tight_layout()
        out_path = label_out / f"{label_name}_page_{page_idx + 1:03d}_fullband_cleanaudio.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
    print(f"Saved {num_pages} page(s) for label '{label_name}' to {label_out}")

if len(scatter_rows) > 0:
    scatter_df = pd.DataFrame(scatter_rows)
    scatter_df = scatter_df[np.isfinite(scatter_df["ridge_med_hz"]) & np.isfinite(scatter_df["fm_range_hz"])].copy()
    if len(scatter_df) > 0:
        labels_sorted = sorted(scatter_df["label"].unique())
        cmap = plt.get_cmap("tab20", max(len(labels_sorted), 1))
        color_map = {label_name: matplotlib.colors.to_hex(cmap(i)) for i, label_name in enumerate(labels_sorted)}
        fig, ax = plt.subplots(figsize=(9, 7))
        for label_name in labels_sorted:
            sub = scatter_df[scatter_df["label"] == label_name]
            if len(sub) == 0:
                continue
            ax.scatter(
                sub["ridge_med_hz"] / 1000.0,
                sub["fm_range_hz"] / 1000.0,
                s=18,
                alpha=0.75,
                label=label_name,
                color=color_map[label_name],
            )
        ax.set_xlabel("Median ridge frequency (kHz)")
        ax.set_ylabel("FM range (kHz)")
        ax.set_title("Call summary: median frequency vs FM range")
        ax.legend(frameon=False, fontsize=8, ncol=2)
        ax.grid(alpha=0.25)
        fig.tight_layout()
        scatter_path = output_folder / "summary_scatter_fullband_cleanaudio_allcalls.png"
        fig.savefig(scatter_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved scatter plot to {scatter_path}")

        fig = plt.figure(figsize=(9, 7))
        ax = fig.add_subplot(111, projection="3d")
        for label_name in labels_sorted:
            sub = scatter_df[scatter_df["label"] == label_name]
            if len(sub) == 0:
                continue
            ax.scatter(
                sub["ridge_med_hz"] / 1000.0,
                sub["fm_range_hz"] / 1000.0,
                sub["duration_s"] * 1000.0,
                s=18,
                alpha=0.75,
                label=label_name,
                color=color_map[label_name],
            )
        ax.set_xlabel("Median ridge frequency (kHz)")
        ax.set_ylabel("FM range (kHz)")
        ax.set_zlabel("Duration (ms)")
        ax.set_title("Call summary (all calls): frequency, FM, duration")
        ax.legend(frameon=False, fontsize=8, ncol=2)
        scatter3d_path = output_folder / "summary_scatter3d_fullband_cleanaudio_allcalls.png"
        fig.tight_layout()
        fig.savefig(scatter3d_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved 3D scatter plot to {scatter3d_path}")

        try:
            import plotly.express as px
            import plotly.graph_objects as go
            import ipywidgets as widgets
            from IPython.display import display

            scatter_df = scatter_df.copy()
            scatter_df["ridge_med_khz"] = scatter_df["ridge_med_hz"] / 1000.0
            scatter_df["fm_range_khz"] = scatter_df["fm_range_hz"] / 1000.0
            scatter_df["duration_ms"] = scatter_df["duration_s"] * 1000.0

            fig3d = px.scatter_3d(
                scatter_df,
                x="ridge_med_khz",
                y="fm_range_khz",
                z="duration_ms",
                color="label",
                hover_name="audio_name",
                custom_data=["preview_path", "audio_name", "label", "ridge_med_khz", "fm_range_khz", "duration_ms"],
                hover_data={
                    "label": True,
                    "ridge_med_khz": ":.2f",
                    "fm_range_khz": ":.2f",
                    "duration_ms": ":.1f",
                },
                labels={
                    "ridge_med_khz": "Median ridge frequency (kHz)",
                    "fm_range_khz": "FM range (kHz)",
                    "duration_ms": "Duration (ms)",
                    "label": "Call type",
                },
                title="Interactive call summary (all calls): frequency, FM, duration",
                color_discrete_map=color_map,
            )
            fig3d.update_traces(marker=dict(size=4, opacity=0.75))
            fig3d.update_layout(
                scene=dict(
                    xaxis_title="Median ridge frequency (kHz)",
                    yaxis_title="FM range (kHz)",
                    zaxis_title="Duration (ms)",
                ),
                margin=dict(l=0, r=0, t=50, b=0),
                clickmode="event+select",
            )
            html_path = output_folder / "summary_scatter3d_fullband_cleanaudio_allcalls.html"
            fig3d.write_html(str(html_path), include_plotlyjs="cdn")
            print(f"Saved interactive 3D scatter to {html_path}")

            figw = go.FigureWidget(fig3d)
            info_html = widgets.HTML()
            preview_out = widgets.Output()

            def _render_preview(preview_path, audio_name, call_label, ridge_med_khz, fm_range_khz, duration_ms):
                info_html.value = (
                    f"<b>{call_label}</b><br>"
                    f"{audio_name}<br>"
                    f"Median ridge: {ridge_med_khz:.2f} kHz<br>"
                    f"FM range: {fm_range_khz:.2f} kHz<br>"
                    f"Duration: {duration_ms:.1f} ms"
                )
                with preview_out:
                    preview_out.clear_output(wait=True)
                    img = plt.imread(preview_path)
                    fig_preview, ax_preview = plt.subplots(figsize=(6.8, 4.4))
                    ax_preview.imshow(img)
                    ax_preview.set_title(f"{call_label} | {audio_name}")
                    ax_preview.axis('off')
                    display(fig_preview)
                    plt.close(fig_preview)

            def _handle_click(trace, points, state):
                if not points.point_inds:
                    return
                idx = points.point_inds[0]
                preview_path, audio_name, call_label, ridge_med_khz, fm_range_khz, duration_ms = trace.customdata[idx]
                _render_preview(preview_path, audio_name, call_label, ridge_med_khz, fm_range_khz, duration_ms)

            for trace in figw.data:
                trace.on_click(_handle_click)

            if len(scatter_df):
                row0 = scatter_df.iloc[0]
                _render_preview(
                    row0['preview_path'],
                    row0['audio_name'],
                    row0['label'],
                    row0['ridge_med_khz'],
                    row0['fm_range_khz'],
                    row0['duration_ms'],
                )

            plot_box = widgets.VBox([
                widgets.HTML('<b>Interactive 3D scatter</b>'),
                figw,
            ], layout=widgets.Layout(width='58%'))
            controls = widgets.VBox([
                widgets.HTML('<b>Spectrogram preview</b>'),
                info_html,
                preview_out,
            ], layout=widgets.Layout(width='42%', padding='0 0 0 24px'))
            display(widgets.HBox([
                plot_box,
                controls,
            ], layout=widgets.Layout(align_items='flex-start', width='100%')))
        except Exception as e:
            print(f"[info] interactive 3D scatter not created: {e}")

print("\nDone with full-band clean_audio-style cell.")


\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10
Found 62 CSV files in \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10
Collected labels:
  alarm: 171 calls
  dense-stack: 126 calls
  dfm-stack: 102 calls
  high-freq: 323 calls
  mnt: 76 calls
  newborn: 13 calls
  noise: 1 calls
  warble: 208 calls

Plotting label 'alarm' with all-call full-band clean_audio mask, n=171


C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\1965304990.py:174: FutureWarning: `binary_opening` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.opening` instead.
  mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\1965304990.py:175: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\1965304990.py:177: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphol

Saved 2 page(s) for label 'alarm' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_allcalls\alarm

Plotting label 'high-freq' with all-call full-band clean_audio mask, n=323


C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\1965304990.py:174: FutureWarning: `binary_opening` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.opening` instead.
  mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\1965304990.py:175: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\1965304990.py:177: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphol

Saved 4 page(s) for label 'high-freq' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_allcalls\high-freq

Plotting label 'warble' with all-call full-band clean_audio mask, n=208


C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\1965304990.py:174: FutureWarning: `binary_opening` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.opening` instead.
  mask = morphology.binary_opening(mask, morphology.disk(VOCAL_DISK_SIZE))
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\1965304990.py:175: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=MIN_OBJ_SIZE)
C:\Users\gg3065\AppData\Local\Temp\ipykernel_36348\1965304990.py:177: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphol

Saved 3 page(s) for label 'warble' to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_allcalls\warble
Saved scatter plot to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_allcalls\summary_scatter_fullband_cleanaudio_allcalls.png
Saved 3D scatter plot to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_allcalls\summary_scatter3d_fullband_cleanaudio_allcalls.png
Saved interactive 3D scatter to \\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_10_viz_harmon_fullband_cleanaudio_allcalls\summary_scatter3d_fullband_cleanaudio_allcalls.html


    'data': [{'custo…


Done with full-band clean_audio-style cell.
